# Fase 2 – Resampling M1 → M15

Questo notebook documenta l'implementazione della **Fase 2** del progetto `AI_microtradingAUX`: generazione delle barre decisionali M15 a partire dal dataset M1 già validato in Fase 1.

Obiettivi principali:

- Conservare le barre M1 originali (in formato Parquet).
- Generare barre M15 tramite resampling controllato.
- Definire correttamente apertura, massimo, minimo, chiusura e volume per ogni barra M15.
- Verificare l'allineamento temporale delle barre risultanti.
- Distinguere il timestamp di apertura da quello di chiusura della candela.
- Eliminare le barre M15 con copertura M1 insufficiente.

**Criterio di completamento**: ogni barra M15 deve contenere solo dati M1 già disponibili al momento della decisione (nessun look-ahead bias).

## Il problema della convenzione temporale MT4/MT5

MetaTrader etichetta ogni barra con il timestamp di **apertura**, non di chiusura: una barra M1 delle 10:00 rappresenta in realtà dati raccolti fino alle 10:01.

Se si costruisce una barra M15 usando direttamente questi timestamp, la barra etichettata "10:00" conterrebbe già dati fino alle 10:15: un caso classico di **look-ahead bias**, perché il modello di trading vedrebbe informazioni non ancora disponibili al momento della decisione.

**Soluzione adottata**: prima di qualunque resampling, ogni timestamp M1 viene traslato in avanti della durata della barra (1 minuto), così l'indice rappresenta l'istante di **chiusura** effettiva. Solo dopo questo shift si procede al resampling M15 con `label='right', closed='right'`, che etichetta ogni barra aggregata con il proprio istante di chiusura.

## Struttura dei file introdotti

- `src/gold_rl/data/resampling.py`: modulo con le funzioni di shift, resampling, validazione OHLC e report diagnostico.
- `config.yaml`: nuova sezione `resampling` con parametri (`target_rule`, `source_bar_duration`, `min_coverage_ratio`, `broker_timestamp_convention`).
- `scripts/build_resampled_data.py`: script che produce gli output finali.
- `tests/test_resampling.py`: test automatici sul comportamento di shift, resampling e validazione.
- `data/processed/xauusd_m1.parquet`: barre M1 con timestamp shiftati a chiusura.
- `data/processed/xauusd_m15.parquet`: barre M15 aggregate, senza look-ahead.
- `reports/resampling_report.json`: riepilogo diagnostico del resampling (finestre totali, mantenute, scartate).

## Modulo `src/gold_rl/data/resampling.py`

Contiene quattro funzioni principali:

1. `shift_open_timestamps_to_close`: sposta l'indice M1 dalla convenzione 'apertura' a 'chiusura'.
2. `resample_ohlcv`: aggrega le barre M1 in barre M15 (o altro timeframe), scartando le finestre con copertura insufficiente.
3. `validate_resampled_ohlcv`: applica alle barre aggregate gli stessi controlli di coerenza OHLC usati in Fase 1.
4. `build_resampling_report`: produce un riepilogo diagnostico (finestre totali/mantenute/scartate, copertura media).

In [ ]:
from typing import Any

import pandas as pd


OHLCV_AGGREGATION = {
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum",
}


def shift_open_timestamps_to_close(
    dataframe: pd.DataFrame,
    bar_duration: str = "1min",
) -> pd.DataFrame:
    """
    Sposta i timestamp M1 da 'apertura candela' (convenzione
    MT4/MT5) a 'chiusura candela'.

    Un file M1 MetaTrader etichetta ogni barra con l'orario di
    apertura. Per evitare look-ahead bias nel resampling e nel
    backtest, l'indice viene traslato in avanti della durata
    della barra, cosi' rappresenta l'istante in cui la barra e'
    effettivamente conclusa e disponibile.
    """
    if dataframe.empty:
        raise ValueError("Il dataset è vuoto.")

    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            "L'indice del DataFrame deve essere un DatetimeIndex."
        )

    shift_delta = pd.to_timedelta(bar_duration)

    shifted = dataframe.copy()
    shifted.index = shifted.index + shift_delta
    shifted.index.name = dataframe.index.name

    return shifted


def resample_ohlcv(
    dataframe: pd.DataFrame,
    target_rule: str = "15min",
    min_coverage_ratio: float = 1.0,
    source_bar_duration: str = "1min",
) -> pd.DataFrame:
    """
    Aggrega barre M1 (con timestamp già spostati a chiusura) in
    barre a timeframe superiore (es. M15).

    Ogni barra risultante è etichettata con il proprio istante di
    chiusura (label='right', closed='right'), cosi' contiene solo
    dati M1 gia' disponibili al momento della decisione.

    Le barre con copertura M1 insufficiente (gap, sessioni
    incomplete) vengono scartate secondo min_coverage_ratio.
    """
    if dataframe.empty:
        raise ValueError("Il dataset è vuoto.")

    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            "L'indice del DataFrame deve essere un DatetimeIndex."
        )

    missing_columns = [
        column
        for column in OHLCV_AGGREGATION
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Colonne OHLCV mancanti per il resampling: {missing_columns}"
        )

    source_delta = pd.to_timedelta(source_bar_duration)
    target_delta = pd.to_timedelta(target_rule)

    expected_bars_per_window = int(
        target_delta / source_delta
    )

    if expected_bars_per_window <= 0:
        raise ValueError(
            "target_rule deve essere un multiplo di "
            "source_bar_duration."
        )

    resampler = dataframe.resample(
        target_rule,
        label="right",
        closed="right",
    )

    resampled = resampler.agg(OHLCV_AGGREGATION)

    bar_counts = resampler["Close"].count()

    coverage_ratio = (
        bar_counts / expected_bars_per_window
    )

    sufficient_coverage_mask = (
        coverage_ratio >= min_coverage_ratio
    )

    resampled = resampled.loc[sufficient_coverage_mask]

    resampled = resampled.dropna(
        subset=list(OHLCV_AGGREGATION)
    )

    resampled["bar_count"] = bar_counts.loc[
        resampled.index
    ].astype("int32")

    resampled["expected_bar_count"] = expected_bars_per_window

    resampled.index.name = dataframe.index.name

    return resampled


def validate_resampled_ohlcv(
    dataframe: pd.DataFrame,
) -> None:
    """
    Verifica la coerenza OHLC delle barre resample-ate, con la
    stessa logica usata per il dataset M1.
    """
    if dataframe.empty:
        raise ValueError("Il dataset resample-ato è vuoto.")

    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            "L'indice del DataFrame deve essere un DatetimeIndex."
        )

    if not dataframe.index.is_monotonic_increasing:
        raise ValueError(
            "Il dataset resample-ato non è ordinato cronologicamente."
        )

    if dataframe.index.has_duplicates:
        raise ValueError(
            "Il dataset resample-ato contiene timestamp duplicati."
        )

    price_columns = ["Open", "High", "Low", "Close"]

    non_positive_price_mask = (
        dataframe[price_columns] <= 0
    ).any(axis=1)

    if non_positive_price_mask.any():
        invalid_count = int(non_positive_price_mask.sum())
        raise ValueError(
            f"Trovate {invalid_count} barre resample-ate con "
            "prezzi nulli o negativi."
        )

    invalid_high_mask = (
        (dataframe["High"] < dataframe["Open"])
        | (dataframe["High"] < dataframe["Close"])
        | (dataframe["High"] < dataframe["Low"])
    )

    invalid_low_mask = (
        (dataframe["Low"] > dataframe["Open"])
        | (dataframe["Low"] > dataframe["Close"])
        | (dataframe["Low"] > dataframe["High"])
    )

    invalid_ohlc_mask = invalid_high_mask | invalid_low_mask

    if invalid_ohlc_mask.any():
        invalid_count = int(invalid_ohlc_mask.sum())
        raise ValueError(
            f"Trovate {invalid_count} barre resample-ate con "
            "OHLC incoerente."
        )

    negative_volume_mask = dataframe["Volume"] < 0

    if negative_volume_mask.any():
        invalid_count = int(negative_volume_mask.sum())
        raise ValueError(
            f"Trovate {invalid_count} barre resample-ate con "
            "volume negativo."
        )


def build_resampling_report(
    source_dataframe: pd.DataFrame,
    resampled_dataframe: pd.DataFrame,
    target_rule: str,
    min_coverage_ratio: float,
) -> dict[str, Any]:
    """
    Costruisce un riepilogo diagnostico del resampling M1 -> M_target.
    """
    total_windows = len(
        source_dataframe.resample(target_rule, label="right", closed="right")
    )

    kept_windows = len(resampled_dataframe)

    dropped_windows = total_windows - kept_windows

    return {
        "source_rows": len(source_dataframe),
        "target_rule": target_rule,
        "min_coverage_ratio": min_coverage_ratio,
        "total_windows": int(total_windows),
        "kept_windows": int(kept_windows),
        "dropped_windows": int(dropped_windows),
        "average_bar_count": float(
            resampled_dataframe["bar_count"].mean()
        ),
        "expected_bar_count": int(
            resampled_dataframe["expected_bar_count"].iloc[0]
        ) if kept_windows > 0 else None,
        "resampled_start": (
            resampled_dataframe.index.min()
            if kept_windows > 0
            else None
        ),
        "resampled_end": (
            resampled_dataframe.index.max()
            if kept_windows > 0
            else None
        ),
    }


## Aggiornamento di `config.yaml`

È stata aggiunta una nuova sezione dedicata al resampling, separata dalla sezione `data` della Fase 1:

```yaml
resampling:
  target_rule: "15min"
  source_bar_duration: "1min"
  min_coverage_ratio: 1.0
  broker_timestamp_convention: "open"
```

Il campo `broker_timestamp_convention` documenta esplicitamente l'assunzione verificata sul CSV (MT4/MT5 usa il timestamp di apertura). Se in futuro si cambia fonte dati, questo campo ricorda di ricontrollare la convenzione prima di applicare lo shift.

## Script `scripts/build_resampled_data.py`

Questo script orchestra l'intera Fase 2:

1. Carica la configurazione da `config.yaml`.
2. Carica e valida il dataset M1 grezzo tramite `load_market_data` (Fase 1).
3. Applica lo shift apertura → chiusura se `broker_timestamp_convention == "open"`.
4. Salva il dataset M1 (timestamp di chiusura) in `data/processed/xauusd_m1.parquet`.
5. Esegue il resampling M15 con `resample_ohlcv`.
6. Valida le barre M15 con `validate_resampled_ohlcv`.
7. Salva il dataset M15 in `data/processed/xauusd_m15.parquet`.
8. Genera un report diagnostico in `reports/resampling_report.json`.

In [ ]:
from pathlib import Path
import json
import sys

import yaml
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
SRC_DIRECTORY = PROJECT_ROOT / "src"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"

if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from gold_rl.data.loader import load_market_data
from gold_rl.data.resampling import (
    build_resampling_report,
    resample_ohlcv,
    shift_open_timestamps_to_close,
    validate_resampled_ohlcv,
)


def load_config() -> dict:
    config_path = PROJECT_ROOT / "config.yaml"

    with config_path.open(mode="r", encoding="utf-8") as config_file:
        config = yaml.safe_load(config_file)

    return config


def main() -> None:
    config = load_config()
    data_config = config["data"]
    resampling_config = config["resampling"]

    processed_directory = PROJECT_ROOT / data_config["processed_directory"]
    processed_directory.mkdir(parents=True, exist_ok=True)

    csv_path = PROJECT_ROOT / data_config["raw_path"]

    print(f"Caricamento M1 grezzo: {csv_path.resolve()}")

    m1_dataframe = load_market_data(
        file_path=csv_path,
        separator=data_config["separator"],
        timestamp_column=data_config["timestamp_column"],
        timestamp_format=data_config["timestamp_format"],
        duplicate_policy=data_config["duplicate_policy"],
    )

    if resampling_config["broker_timestamp_convention"] == "open":
        print("Convenzione broker: timestamp di apertura. Applico lo shift a chiusura.")
        m1_shifted = shift_open_timestamps_to_close(
            dataframe=m1_dataframe,
            bar_duration=resampling_config["source_bar_duration"],
        )
    else:
        print("Convenzione broker: timestamp già di chiusura. Nessun shift applicato.")
        m1_shifted = m1_dataframe.copy()

    m1_output_path = processed_directory / "xauusd_m1.parquet"
    m1_shifted.to_parquet(m1_output_path)
    print(f"Salvato M1 (timestamp di chiusura): {m1_output_path.resolve()}")

    m15_dataframe = resample_ohlcv(
        dataframe=m1_shifted,
        target_rule=resampling_config["target_rule"],
        min_coverage_ratio=resampling_config["min_coverage_ratio"],
        source_bar_duration=resampling_config["source_bar_duration"],
    )

    validate_resampled_ohlcv(m15_dataframe)

    m15_output_path = processed_directory / "xauusd_m15.parquet"
    m15_dataframe.to_parquet(m15_output_path)
    print(f"Salvato M15: {m15_output_path.resolve()}")

    report = build_resampling_report(
        source_dataframe=m1_shifted,
        resampled_dataframe=m15_dataframe,
        target_rule=resampling_config["target_rule"],
        min_coverage_ratio=resampling_config["min_coverage_ratio"],
    )

    report_path = REPORTS_DIRECTORY / "resampling_report.json"
    with report_path.open(mode="w", encoding="utf-8") as report_file:
        json.dump(report, report_file, indent=4, default=str)

    print(f"Report di resampling: {report_path.resolve()}")


if __name__ == "__main__":
    main()


## Test automatici `tests/test_resampling.py`

I test verificano che:

- lo shift sposti correttamente i timestamp M1;
- ogni barra M15 sia costruita solo con dati M1 con timestamp minore o uguale all'etichetta della barra (nessun look-ahead);
- le finestre M15 con copertura M1 incompleta vengano scartate;
- `validate_resampled_ohlcv` rifiuti barre con OHLC incoerente.

In [ ]:
import pandas as pd
import pytest

from gold_rl.data.resampling import (
    resample_ohlcv,
    shift_open_timestamps_to_close,
    validate_resampled_ohlcv,
)


def build_m1_sample() -> pd.DataFrame:
    index = pd.date_range(
        "2024-01-01 10:00",
        periods=30,
        freq="1min",
    )

    data = {
        "Open": range(30),
        "High": [value + 1 for value in range(30)],
        "Low": [value - 1 for value in range(30)],
        "Close": range(30),
        "Volume": [1.0] * 30,
    }

    return pd.DataFrame(data, index=index)


def test_shift_moves_timestamps_forward():
    m1 = build_m1_sample()
    shifted = shift_open_timestamps_to_close(m1, bar_duration="1min")

    assert shifted.index[0] == m1.index[0] + pd.Timedelta(minutes=1)


def test_resample_uses_only_past_data():
    m1 = build_m1_sample()
    shifted = shift_open_timestamps_to_close(m1, bar_duration="1min")

    m15 = resample_ohlcv(
        shifted,
        target_rule="15min",
        min_coverage_ratio=1.0,
        source_bar_duration="1min",
    )

    first_bar_label = m15.index[0]
    contributing_m1 = shifted.loc[shifted.index <= first_bar_label]

    assert contributing_m1.index.max() == first_bar_label
    assert (shifted.loc[shifted.index > first_bar_label].index
            .isin(contributing_m1.index)).sum() == 0


def test_incomplete_window_is_dropped():
    m1 = build_m1_sample().iloc[:10]
    shifted = shift_open_timestamps_to_close(m1, bar_duration="1min")

    m15 = resample_ohlcv(
        shifted,
        target_rule="15min",
        min_coverage_ratio=1.0,
        source_bar_duration="1min",
    )

    assert m15.empty


def test_validate_resampled_ohlcv_rejects_invalid_bars():
    invalid = pd.DataFrame(
        {
            "Open": [10.0],
            "High": [9.0],
            "Low": [8.0],
            "Close": [10.0],
            "Volume": [1.0],
            "bar_count": [15],
            "expected_bar_count": [15],
        },
        index=pd.date_range("2024-01-01 10:15", periods=1, freq="15min"),
    )

    with pytest.raises(ValueError):
        validate_resampled_ohlcv(invalid)


## Sequenza operativa

1. Creare `src/gold_rl/data/resampling.py` con il codice della sezione precedente.
2. Aggiungere la sezione `resampling` a `config.yaml`.
3. Creare `scripts/build_resampled_data.py`.
4. Creare `tests/test_resampling.py`.
5. Eseguire i test:

```bash
pytest tests/test_resampling.py -v
```

6. Eseguire lo script di produzione:

```bash
python scripts/build_resampled_data.py
```

7. Verificare gli output generati:
   - `data/processed/xauusd_m1.parquet`
   - `data/processed/xauusd_m15.parquet`
   - `reports/resampling_report.json`

## Uso dei file M5/M15/M30 già disponibili come controllo incrociato

I CSV M5/M15/M30 forniti separatamente non sostituiscono la Fase 2: vengono usati come **verifica indipendente**, confrontando un campione di barre M15 generate da `xauusd_m15.parquet` con le barre corrispondenti nel file M15 esterno.

Se emergono differenze sistematiche di 15 minuti sull'indice, ciò indica che il file esterno etichetta le barre M15 con l'orario di apertura (non di chiusura) — un'informazione utile da annotare nella relazione di progetto, insieme al report `resampling_report.json`.

## Conclusione

La Fase 2 garantisce che ogni barra M15 contenga esclusivamente dati M1 già disponibili al momento della decisione, eliminando il rischio di look-ahead bias legato alla convenzione MT4/MT5 sui timestamp di apertura. Le barre M15 con copertura M1 insufficiente vengono scartate, e la coerenza OHLC viene verificata sia sul dataset M1 di partenza sia sulle barre M15 risultanti.